In [0]:
ENV = dbutils.widgets.get("env") if dbutils.widgets.get("env") else "dev"

In [0]:
from pyspark.sql import functions as F
from datetime import datetime 

# config
STORAGE_ACCOUNT = "nyctaxiistorage"
LANDING_PATH = f"abfss://landing@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
BRONZE_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
BRONZE_CATALOG = "dev"
BRONZE_SCHEMA = "bronze"
BRONZE_TABLE = "yellow_taxi_raw"
SOURCE_FILE = "yellow_tripdata_2023-01.parquet"

In [0]:
def history_loader(source_path, table_name):
  print(f"Starting historical load from {source_path}")
  df = spark.read.parquet(source_path)
    
  # Add metadata columns
  df = df.withColumn("ingestion_timestamp", F.current_timestamp()).withColumn("source_file", F.lit(SOURCE_FILE))

  # write to delta table
  (df.write.format("delta").mode("overwrite")).saveAsTable(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{table_name}")

  print(f"Historical load complete. Rows loaded: {df.count()}")
  return df

In [0]:
def batch_ingestion(source_path, table_name):
    """
    Batch ingestion function.
    Reads new files from landing and appends to Bronze Delta table.
    """
    print(f"Starting batch ingestion from {source_path}")
    
    df = spark.read.parquet(source_path)
    
    # Add metadata columns
    df = df.withColumn("ingestion_timestamp", F.current_timestamp()) \
           .withColumn("source_file", F.lit(SOURCE_FILE))
    
    # Append to existing Delta table
    (df.write
       .format("delta")
       .mode("append")
       .saveAsTable(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{table_name}"))
    
    print(f"Batch ingestion complete. Rows loaded: {df.count()}")
    return df

In [0]:
def stream_ingestion(source_path, checkpoint_path, table_name):
    print(f"Starting streaming ingestion from {source_path}")
    df = (spark.readStream
                 .format("cloudFiles")
                 .option("cloudFiles.format", "parquet")
                 .option("cloudFiles.schemaLocation", checkpoint_path)
                 .load(source_path))
    
    # Add metadata columns
    df_stream = df_stream \
        .withColumn("ingestion_timestamp", F.current_timestamp()) \
        .withColumn("source_file", F.input_file_name())

    # Write stream to Delta table with checkpointing
    query = (df_stream.writeStream
             .format("delta")
             .option("checkpointLocation", checkpoint_path)
             .outputMode("append")
             .toTable(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{table_name}"))
    
    return query

In [0]:
# run historical load first
full_path = LANDING_PATH + SOURCE_FILE
df = history_loader(full_path, BRONZE_TABLE)
df.printSchema()